# Tugas Praktikum 6 – Classical Synchronization Problems

**Nama:** Najwa Azzahra  
**NIM:** 2411513004

## 1. Dining Philosophers dengan N Berbeda

Setiap filsuf membutuhkan dua garpu yang berada di sebelahnya untuk makan. Semaphore digunakan untuk mengunci setiap garpu agar tidak digunakan oleh dua filsuf secara bersamaan. Program mengukur throughput berdasarkan jumlah total aktivitas makan dibagi waktu simulasi.

In [1]:
import threading
import time

def simulasi_dining(n, putaran=25):
    garpu = [threading.Semaphore(1) for _ in range(n)]
    total_makan = [0] * n
    mulai = time.perf_counter()

    def filsuf(i):
        kiri = i
        kanan = (i + 1) % n
        for _ in range(putaran):
            # Urutan penguncian dibuat konsisten untuk mencegah deadlock.
            pertama, kedua = (garpu[kiri], garpu[kanan]) if i % 2 == 0 else (garpu[kanan], garpu[kiri])
            with pertama:
                with kedua:
                    time.sleep(0.002)
                    total_makan[i] += 1

    threads = [threading.Thread(target=filsuf, args=(i,)) for i in range(n)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    durasi = time.perf_counter() - mulai
    jumlah = sum(total_makan)
    throughput = jumlah / durasi
    return jumlah, durasi, throughput

hasil_dining = {}
for n in (3, 7, 10):
    jumlah, durasi, throughput = simulasi_dining(n)
    hasil_dining[n] = throughput
    print(f"N={n:2d} | total makan={jumlah:3d} | waktu={durasi:.3f} detik | throughput={throughput:.2f} makan/detik")

N= 3 | total makan= 75 | waktu=0.159 detik | throughput=472.75 makan/detik
N= 7 | total makan=175 | waktu=0.158 detik | throughput=1104.38 makan/detik


N=10 | total makan=250 | waktu=0.166 detik | throughput=1508.27 makan/detik


**Analisis:** Throughput tidak selalu naik secara linear terhadap jumlah filsuf. Penambahan filsuf memang menambah jumlah aktivitas makan yang dapat berlangsung, tetapi setiap filsuf tetap harus menunggu dua garpu dan beberapa filsuf saling berebut sumber daya yang sama. Karena itu, peningkatan jumlah filsuf juga menambah persaingan, waktu tunggu, dan overhead thread. Nilai throughput dapat sedikit berbeda setiap kali program dijalankan karena penjadwalan thread.

## 2. Readers-Writers dengan Prioritas Writer

Pada solusi ini, writer mendapat prioritas. Ketika ada writer yang sedang menunggu, reader baru tidak diizinkan masuk. Reader yang sudah mulai membaca tetap boleh menyelesaikan prosesnya. Kondisi ini dapat menyebabkan reader baru mengalami starvation apabila writer terus berdatangan.

In [2]:
import threading
import time

class ReadersWritersPrioritasWriter:
    def __init__(self):
        self.kondisi = threading.Condition()
        self.active_readers = 0
        self.active_writer = False
        self.waiting_writers = 0
        self.data = 0

    def mulai_baca(self):
        with self.kondisi:
            while self.active_writer or self.waiting_writers > 0:
                self.kondisi.wait()
            self.active_readers += 1

    def selesai_baca(self):
        with self.kondisi:
            self.active_readers -= 1
            if self.active_readers == 0:
                self.kondisi.notify_all()

    def mulai_tulis(self):
        with self.kondisi:
            self.waiting_writers += 1
            while self.active_writer or self.active_readers > 0:
                self.kondisi.wait()
            self.waiting_writers -= 1
            self.active_writer = True

    def selesai_tulis(self):
        with self.kondisi:
            self.active_writer = False
            self.kondisi.notify_all()

rw = ReadersWritersPrioritasWriter()
log_aktivitas = []
log_lock = threading.Lock()

def reader(nama, jeda=0):
    time.sleep(jeda)
    rw.mulai_baca()
    with log_lock:
        log_aktivitas.append(f"{nama} mulai membaca")
    time.sleep(0.15)
    with log_lock:
        log_aktivitas.append(f"{nama} selesai membaca (data={rw.data})")
    rw.selesai_baca()

def writer(nama, jeda=0):
    time.sleep(jeda)
    rw.mulai_tulis()
    with log_lock:
        log_aktivitas.append(f"{nama} mulai menulis")
    rw.data += 1
    time.sleep(0.15)
    with log_lock:
        log_aktivitas.append(f"{nama} selesai menulis (data={rw.data})")
    rw.selesai_tulis()

# Reader pertama mulai lebih dahulu. Writer datang saat reader sedang membaca.
# Reader baru datang setelah writer menunggu sehingga harus menunggu writer selesai.
threads = [
    threading.Thread(target=reader, args=("Reader-1", 0.00)),
    threading.Thread(target=writer, args=("Writer-1", 0.04)),
    threading.Thread(target=reader, args=("Reader-2", 0.06)),
    threading.Thread(target=writer, args=("Writer-2", 0.08)),
]
for t in threads:
    t.start()
for t in threads:
    t.join()

for aktivitas in log_aktivitas:
    print("-", aktivitas)

- Reader-1 mulai membaca
- Reader-1 selesai membaca (data=0)
- Writer-2 mulai menulis
- Writer-2 selesai menulis (data=1)
- Writer-1 mulai menulis
- Writer-1 selesai menulis (data=2)
- Reader-2 mulai membaca
- Reader-2 selesai membaca (data=2)


**Analisis starvation:** Reader baru berisiko mengalami starvation karena setiap writer yang menunggu akan menghalangi reader baru. Jika writer datang terus-menerus, jumlah writer yang menunggu mungkin tidak pernah menjadi nol sehingga reader harus terus menunggu. Namun, reader yang sudah masuk sebelum writer menunggu tetap dapat menyelesaikan proses membaca.

## 3. Sleeping Barber dengan Dua Tukang Cukur

Dua tukang cukur menggunakan satu antrean pelanggan yang sama. Kapasitas ruang tunggu dibatasi oleh jumlah kursi. Pelanggan masuk ke antrean jika masih tersedia tempat; jika antrean penuh, pelanggan ditolak. Tukang cukur yang selesai lebih dahulu akan mengambil pelanggan berikutnya dari antrean.

In [3]:
import threading
import queue
import time

jumlah_kursi = 4
antrean = queue.Queue(maxsize=jumlah_kursi)
jumlah_tukang_cukur = 2
pelanggan_dilayani = []
pelanggan_ditolak = []
hasil_lock = threading.Lock()
STOP = object()

def tukang_cukur(nama):
    while True:
        pelanggan = antrean.get()
        if pelanggan is STOP:
            antrean.task_done()
            break
        print(f"{nama} melayani {pelanggan}")
        time.sleep(0.12)  # simulasi waktu mencukur
        with hasil_lock:
            pelanggan_dilayani.append((pelanggan, nama))
        print(f"{nama} selesai melayani {pelanggan}")
        antrean.task_done()

barber_threads = [
    threading.Thread(target=tukang_cukur, args=(f"Tukang Cukur-{i+1}",))
    for i in range(jumlah_tukang_cukur)
]
for t in barber_threads:
    t.start()

# Pelanggan datang berurutan. Pelanggan yang tidak tertampung akan ditolak.
for i in range(1, 11):
    nama = f"Pelanggan-{i}"
    try:
        antrean.put_nowait(nama)
        print(f"{nama} masuk antrean")
    except queue.Full:
        pelanggan_ditolak.append(nama)
        print(f"{nama} ditolak karena ruang tunggu penuh")
    time.sleep(0.035)

antrean.join()
for _ in range(jumlah_tukang_cukur):
    antrean.put(STOP)
for t in barber_threads:
    t.join()

print("\nRingkasan:")
print("Jumlah dilayani:", len(pelanggan_dilayani))
print("Jumlah ditolak:", len(pelanggan_ditolak))
print("Daftar pelanggan ditolak:", pelanggan_ditolak)

Pelanggan-1 masuk antreanTukang Cukur-1 melayani Pelanggan-1

Pelanggan-2 masuk antrean
Tukang Cukur-2 melayani Pelanggan-2
Pelanggan-3 masuk antrean
Pelanggan-4 masuk antrean
Tukang Cukur-1 selesai melayani Pelanggan-1
Tukang Cukur-1 melayani Pelanggan-3
Pelanggan-5 masuk antrean
Tukang Cukur-2 selesai melayani Pelanggan-2
Tukang Cukur-2 melayani Pelanggan-4
Pelanggan-6 masuk antrean


Pelanggan-7 masuk antrean
Tukang Cukur-1 selesai melayani Pelanggan-3
Tukang Cukur-1 melayani Pelanggan-5
Pelanggan-8 masuk antrean
Tukang Cukur-2 selesai melayani Pelanggan-4
Tukang Cukur-2 melayani Pelanggan-6
Pelanggan-9 masuk antrean
Pelanggan-10 masuk antrean
Tukang Cukur-1 selesai melayani Pelanggan-5
Tukang Cukur-1 melayani Pelanggan-7
Tukang Cukur-2 selesai melayani Pelanggan-6
Tukang Cukur-2 melayani Pelanggan-8


Tukang Cukur-1 selesai melayani Pelanggan-7
Tukang Cukur-1 melayani Pelanggan-9
Tukang Cukur-2 selesai melayani Pelanggan-8
Tukang Cukur-2 melayani Pelanggan-10
Tukang Cukur-1 selesai melayani Pelanggan-9
Tukang Cukur-2 selesai melayani Pelanggan-10

Ringkasan:
Jumlah dilayani: 10
Jumlah ditolak: 0
Daftar pelanggan ditolak: []


**Penjelasan:** Antrean bersama memastikan pelanggan dilayani berdasarkan urutan masuk, sedangkan dua thread tukang cukur mengambil pelanggan dari antrean yang sama. Tukang cukur yang sudah selesai akan mengambil pelanggan berikutnya. Dengan demikian, kedua tukang cukur dapat bekerja secara bersamaan tanpa melayani pelanggan yang sama. Semaphore tambahan dapat digunakan untuk menghitung jumlah kursi kosong, tetapi pada simulasi ini kapasitas antrean dikelola oleh `queue.Queue`.

## 4. Refleksi Singkat

Menurut saya, masalah Readers-Writers paling mirip dengan masalah nyata pada pengembangan software, khususnya database. Beberapa pengguna dapat membaca data secara bersamaan, tetapi ketika ada proses yang mengubah data, akses perlu diatur agar data tidak berubah saat sedang dibaca. Contohnya, pada aplikasi perbankan, banyak pengguna dapat melihat saldo, sedangkan transaksi penarikan atau transfer harus mendapat akses tulis secara terkontrol. Prioritas writer dapat mempercepat proses perubahan data, tetapi jika transaksi tulis terus berdatangan, permintaan baca baru berisiko menunggu terlalu lama (starvation). Karena itu, sistem perlu menerapkan mekanisme sinkronisasi dan kebijakan antrean yang adil agar konsistensi data tetap terjaga serta setiap permintaan memperoleh kesempatan diproses.

## Kesimpulan

Ketiga simulasi menunjukkan pentingnya sinkronisasi untuk mengatur penggunaan sumber daya bersama. Dining Philosophers memperlihatkan persaingan dalam penggunaan garpu, Readers-Writers menunjukkan dampak kebijakan prioritas terhadap starvation, sedangkan Sleeping Barber memperlihatkan pengelolaan antrean dengan lebih dari satu pelayan.